# 03 - Join Bike Counts + Weather

Joins each station's 15-minute bike-count series with the combined hourly
DWD weather record into one per-station dataset, ready for later
feature-engineering / modeling notebooks.

This notebook only orchestrates calls into
`src/muenster_bike_forecast/data/join.py` (plus the existing
`bike_counts.py` / `weather.py` loaders) and reports results; it does
**not** hit the network — it consumes the already-saved raw CSVs under
`data/raw/bike_counts/` and `data/raw/weather/` (see `01_fetch_bike_counts`
and `02_fetch_weather`).

## Design decisions (see `join.py` docstrings for full detail)

- **Timezone handling.** Bike-count `datetime` values are naive
  Europe/Berlin local time; weather `timestamp` values are UTC-aware.
  Before joining, bike timestamps are localized to Europe/Berlin with
  `ambiguous="NaT", nonexistent="NaT"` and converted to UTC — the DST
  fall-back/spring-forward edge-case timestamps become explicit nulls
  (never silently guessed at), and their count is reported below.
- **The join.** For each station, its 15-minute rows are joined against
  the combined hourly weather frame with `pandas.merge_asof(...,
  direction="backward")`: each bike-count row gets the most recent *past*
  weather reading, never a future one (this is a forecasting pipeline —
  a future weather value leaking onto a past row would invalidate any
  model trained on the result). A 2-hour tolerance means a bike-count row
  far past a weather data gap is left with null weather columns instead
  of being matched to stale data. Coverage (the null-weather fraction) is
  reported per station below.
- **Weather column collisions.** All three weather parameters
  (`air_temperature`, `precipitation`, `wind`) have their own
  `quality_level` column; `combine_weather_parameters` prefixes colliding
  names with the parameter key so none of them silently overwrite another.


In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.bike_counts import load_station_data
from muenster_bike_forecast.data.weather import (
    DEFAULT_STATION_ID,
    PARAMETER_SPECS,
    load_weather_data,
)
from muenster_bike_forecast.data.join import (
    JoinError,
    combine_weather_parameters,
    join_station_weather,
    localize_bike_timestamps,
    summarize_dst_edge_cases,
    summarize_weather_coverage,
)

BIKE_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "bike_counts"
WEATHER_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "weather"
JOINED_DIR = PROJECT_ROOT / "data" / "raw" / "joined"
JOINED_DIR.mkdir(parents=True, exist_ok=True)

# Bike-count rows further than this past the nearest weather reading (e.g.
# spanning a weather data gap) get null weather columns instead of being
# matched to stale data - see `join_station_weather`.
WEATHER_TOLERANCE = pd.Timedelta(hours=2)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print(f"Weather station: {DEFAULT_STATION_ID}")
print(f"Bike-count raw dir: {BIKE_RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Weather raw dir: {WEATHER_RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Output dir: {JOINED_DIR.relative_to(PROJECT_ROOT)}")

Weather station: 01766
Bike-count raw dir: data\raw\bike_counts
Weather raw dir: data\raw\weather
Output dir: data\raw\joined


## 1. Load and combine the raw weather data

Loads the three saved per-parameter weather CSVs and combines them into
one wide hourly DataFrame (joined on `timestamp`, since all three describe
the same single weather station).

In [2]:
weather_frames = {
    parameter: load_weather_data(
        WEATHER_RAW_DIR / f"dwd_{parameter}_{DEFAULT_STATION_ID}.csv"
    )
    for parameter in PARAMETER_SPECS
}
for parameter, df in weather_frames.items():
    print(f"{parameter}: {len(df):,} rows, {df['timestamp'].min()} .. {df['timestamp'].max()}")

combined_weather = combine_weather_parameters(weather_frames)
print(f"\nCombined weather: {len(combined_weather):,} rows, "
      f"{combined_weather['timestamp'].min()} .. {combined_weather['timestamp'].max()}")
combined_weather.head()

air_temperature: 323,344 rows, 1989-10-01 07:00:00+00:00 .. 2026-08-20 23:00:00+00:00
precipitation: 270,420 rows, 1995-09-01 00:00:00+00:00 .. 2026-08-20 23:00:00+00:00
wind: 389,637 rows, 1982-01-01 00:00:00+00:00 .. 2026-08-20 23:00:00+00:00

Combined weather: 390,340 rows, 1982-01-01 00:00:00+00:00 .. 2026-08-20 23:00:00+00:00


,station_id,timestamp,quality_level,air_temperature_c,relative_humidity_pct,precipitation_quality_level,precipitation_mm,precipitation_indicator,precipitation_form,wind_quality_level,wind_speed_ms,wind_direction_deg
0,01766,1982-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,2.1,200.0
1,01766,1982-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,1.4,240.0
2,01766,1982-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,1.9,130.0
3,01766,1982-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,2.0,130.0
4,01766,1982-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,1.9,100.0


## 2. Load raw bike-count data for every station

Every CSV under `data/raw/bike_counts/` except the non-station files
(`stations.json`, `missing_intervals.csv`, `station_locations.csv`) is one station's data, as
written by `save_station_data` in `01_fetch_bike_counts`.

In [3]:
NON_STATION_FILES = {"stations.json", "missing_intervals.csv", "station_locations.csv"}
station_csv_paths = sorted(
    p for p in BIKE_RAW_DIR.glob("*.csv") if p.name not in NON_STATION_FILES
)
station_ids = [p.stem for p in station_csv_paths]
print(f"{len(station_ids)} stations found: {station_ids}")

bike_frames = {station_id: load_station_data(path)
               for station_id, path in zip(station_ids, station_csv_paths)}
{sid: len(df) for sid, df in bike_frames.items()}

23 stations found: ['100020113', '100031297', '100031300', '100034978', '100034980', '100034981', '100034982', '100034983', '100035541', '100053305', '300037405', '300037544', '300037920', '300037925', '300037926', '300037928', '300037931', '300037932', '300037933', '300037936', '300038855', '300039328', '300039331']


C:\Users\FloKI\Documents\Daten\muenster-bike-traffic-forecast\src\muenster_bike_forecast\data\bike_counts.py:753: DtypeWarning: Columns (0: 300037931-status, 1: 353413846-status, 2: 353413847-status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


{'100020113': 125216,
 '100031297': 121935,
 '100031300': 226084,
 '100034978': 123684,
 '100034980': 123768,
 '100034981': 123684,
 '100034982': 123316,
 '100034983': 123674,
 '100035541': 123660,
 '100053305': 120810,
 '300037405': 94268,
 '300037544': 90974,
 '300037920': 91764,
 '300037925': 85243,
 '300037926': 92417,
 '300037928': 91401,
 '300037931': 92735,
 '300037932': 60101,
 '300037933': 87571,
 '300037936': 89717,
 '300038855': 46132,
 '300039328': 90130,
 '300039331': 92953}

## 3. Localize bike-count timestamps to UTC

Converts each station's naive Europe/Berlin `datetime` values to a UTC-aware
`timestamp` column. Rows that fall on a DST ambiguous (autumn fall-back) or
nonexistent (spring forward-jump) local time become explicit nulls, counted
below rather than silently dropped.

In [4]:
localized_frames: dict[str, pd.DataFrame] = {}
dst_rows = []

for station_id, df in bike_frames.items():
    localized = localize_bike_timestamps(df)
    localized_frames[station_id] = localized
    summary = summarize_dst_edge_cases(localized)
    dst_rows.append({"station_id": station_id, **summary})

dst_report = pd.DataFrame(dst_rows).set_index("station_id")
dst_report

,n_rows,n_dst_edge_case,pct_dst_edge_case
station_id,,,
100020113,125216,12,0.009583
100031297,121935,12,0.009841
100031300,226084,20,0.008846
100034978,123684,12,0.009702
100034980,123768,12,0.009696
100034981,123684,12,0.009702
100034982,123316,12,0.009731
100034983,123674,12,0.009703
100035541,123660,12,0.009704


In [5]:
total_rows = int(dst_report["n_rows"].sum())
total_dst_edge_cases = int(dst_report["n_dst_edge_case"].sum())
print(
    f"Total rows across all stations: {total_rows:,}\n"
    f"Total DST edge-case rows (ambiguous/nonexistent local time): "
    f"{total_dst_edge_cases} "
    f"({100 * total_dst_edge_cases / total_rows:.5f}% of all rows)"
)

Total rows across all stations: 2,441,237
Total DST edge-case rows (ambiguous/nonexistent local time): 225 (0.00922% of all rows)


## 4. Join each station against the combined hourly weather

For every station, as-of joins its localized 15-minute rows against
`combined_weather` (`direction="backward"`, `tolerance=WEATHER_TOLERANCE`),
saves the result under `data/raw/joined/<station_id>.csv`, and records the
weather-coverage stats (fraction of rows with no matched weather
reading).

In [6]:
coverage_rows = []
joined_frames: dict[str, pd.DataFrame] = {}

for station_id, localized in localized_frames.items():
    joined = join_station_weather(
        localized, combined_weather, tolerance=WEATHER_TOLERANCE
    )
    joined_frames[station_id] = joined
    out_path = JOINED_DIR / f"{station_id}.csv"
    joined.to_csv(out_path, index=False)
    coverage_rows.append(summarize_weather_coverage(joined, station_id=station_id))
    print(f"{station_id}: {len(joined):,} rows -> {out_path.relative_to(PROJECT_ROOT)}")

coverage_report = pd.DataFrame(coverage_rows).set_index("station_id")
coverage_report

100020113: 125,216 rows -> data\raw\joined\100020113.csv


100031297: 121,935 rows -> data\raw\joined\100031297.csv


100031300: 226,084 rows -> data\raw\joined\100031300.csv


100034978: 123,684 rows -> data\raw\joined\100034978.csv


100034980: 123,768 rows -> data\raw\joined\100034980.csv


...

300037936: 89,717 rows -> data\raw\joined\300037936.csv


300038855: 46,132 rows -> data\raw\joined\300038855.csv


300039328: 90,130 rows -> data\raw\joined\300039328.csv


300039331: 92,953 rows -> data\raw\joined\300039331.csv


,n_rows,n_missing_weather,pct_missing_weather
station_id,,,
100020113,125216,12,0.009583
100031297,121935,12,0.009841
100031300,226084,20,0.008846
100034978,123684,12,0.009702
100034980,123768,12,0.009696
100034981,123684,12,0.009702
100034982,123316,12,0.009731
100034983,123674,12,0.009703
100035541,123660,12,0.009704


In [7]:
total_joined_rows = int(coverage_report["n_rows"].sum())
total_missing_weather = int(coverage_report["n_missing_weather"].sum())
print(
    f"Overall weather coverage across all stations: "
    f"{total_joined_rows - total_missing_weather:,} / {total_joined_rows:,} rows matched "
    f"({100 * (1 - total_missing_weather / total_joined_rows):.3f}%)\n"
    f"Per-station weather-coverage range: "
    f"{100 - coverage_report['pct_missing_weather'].max():.3f}% .. "
    f"{100 - coverage_report['pct_missing_weather'].min():.3f}%"
)

Overall weather coverage across all stations: 2,441,012 / 2,441,237 rows matched (99.991%)
Per-station weather-coverage range: 99.990% .. 99.992%


## 5. First look at the joined data

Head, dtypes, and date range for one example station, plus the overall
date range covered across all joined stations.

In [8]:
example_station_id = station_ids[0]
example = joined_frames[example_station_id]
print(f"Example station: {example_station_id}")
print(f"Shape: {example.shape}")
print(f"Date range: {example['timestamp'].min()} .. {example['timestamp'].max()}")
example.dtypes

Example station: 100020113
Shape: (125216, 22)
Date range: 2022-12-31 23:00:00+00:00 .. 2026-08-20 21:45:00+00:00


station_id                                           int64
datetime                                    datetime64[us]
100020113 (Wolbecker Straße)                         int64
101020113 (FR stdteinwärts)                        float64
102020113 (FR stadtauswärts)                         int64
100020113-status                                     int64
101020113-status                                     int64
102020113-status                                     int64
101020113 (FR stadteinwärts)                       float64
timestamp                              datetime64[us, UTC]
weather_station_id                                  object
weather_quality_level                               object
weather_air_temperature_c                           object
weather_relative_humidity_pct                       object
weather_precipitation_quality_level                 object
weather_precipitation_mm                            object
weather_precipitation_indicator                     obje

In [9]:
example.head()

,station_id,datetime,100020113 (Wolbecker Straße),101020113 (FR stdteinwärts),102020113 (FR stadtauswärts),100020113-status,101020113-status,102020113-status,101020113 (FR stadteinwärts),timestamp,...,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,weather_timestamp
0,100020113,2023-01-01 00:00:00,1,1.0,0,0,0,0,NaN,2022-12-31 23:00:00+00:00,...,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2022-12-31 23:00:00+00:00
1,100020113,2023-01-01 00:15:00,15,7.0,8,0,0,0,NaN,2022-12-31 23:15:00+00:00,...,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2022-12-31 23:00:00+00:00
2,100020113,2023-01-01 00:30:00,16,11.0,5,0,0,0,NaN,2022-12-31 23:30:00+00:00,...,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2022-12-31 23:00:00+00:00
3,100020113,2023-01-01 00:45:00,21,10.0,11,0,0,0,NaN,2022-12-31 23:45:00+00:00,...,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2022-12-31 23:00:00+00:00
4,100020113,2023-01-01 01:00:00,35,16.0,19,0,0,0,NaN,2023-01-01 00:00:00+00:00,...,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,2023-01-01 00:00:00+00:00


In [10]:
overall_range = pd.DataFrame(
    {
        "station_id": station_ids,
        "first_timestamp": [joined_frames[s]["timestamp"].min() for s in station_ids],
        "last_timestamp": [joined_frames[s]["timestamp"].max() for s in station_ids],
        "n_rows": [len(joined_frames[s]) for s in station_ids],
    }
).set_index("station_id")
overall_range

,first_timestamp,last_timestamp,n_rows
station_id,,,
100020113,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,125216
100031297,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,121935
100031300,2019-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,226084
100034978,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123684
100034980,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123768
100034981,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123684
100034982,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123316
100034983,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123674
100035541,2022-12-31 23:00:00+00:00,2026-08-20 21:45:00+00:00,123660
